In [15]:
%pip install jinja2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

print("Loading data")
df = pd.read_pickle("data_processed.pkl")
df['text_content'] = df['text_content'].fillna("")

# We define if a film was liked by getting films which rating was above 3.5
df['is_liked'] = (df['rating'] >= 3.5).astype(int)

# We save columns into global variables that we will use during training.
X = df['text_content']
y = df['is_liked']

print(f"Size of the Dataset: {len(df)}")
print(f"Distribution of reviews - Positives (1): {y.sum()} | Negatives (0): {len(df) - y.sum()}")

Loading data
Size of the Dataset: 25000
Distribution of reviews - Positives (1): 18802 | Negatives (0): 6198


In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"• Training set (X_train): {X_train.shape[0]}")
print(f"• Test set (X_test): {X_test.shape[0]} ")

• Training set (X_train): 20000
• Test set (X_test): 5000 


In [3]:
print("Vectorizing with Bag of Words (BoW)...")
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# 1. Naive Bayes with BoW
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
preds_nb_bow = nb_bow.predict(X_test_bow)

# 2. Logistic regression with BoW
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)
preds_lr_bow = lr_bow.predict(X_test_bow)

print("Models trained with Bag of Words.")

Vectorizing with Bag of Words (BoW)...
Models trained with Bag of Words.


In [4]:
print("Vectorizing with TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 1. Naive Bayes with TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
preds_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

# 2. Logistic regression with TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
preds_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

print("Models trained with TF-IDF.")

Vectorizing with TF-IDF...
Models trained with TF-IDF.


In [ ]:

results = [
    {"Vectorization": "Bag of Words (BoW)", "Model": "Naive Bayes", 
     "Accuracy": accuracy_score(y_test, preds_nb_bow), "F1-Score": f1_score(y_test, preds_nb_bow)},
    {"Vectorization": "Bag of Words (BoW)", "Model": "Logistic Regression", 
     "Accuracy": accuracy_score(y_test, preds_lr_bow), "F1-Score": f1_score(y_test, preds_lr_bow)},
    {"Vectorization": "TF-IDF", "Model": "Naive Bayes", 
     "Accuracy": accuracy_score(y_test, preds_nb_tfidf), "F1-Score": f1_score(y_test, preds_nb_tfidf)},
    {"Vectorization": "TF-IDF", "Model": "Logistic Regression", 
     "Accuracy": accuracy_score(y_test, preds_lr_tfidf), "F1-Score": f1_score(y_test, preds_lr_tfidf)},
]

df_results = pd.DataFrame(results)

print("="*60)
print("🏆           MODEL COMPARISON REPORT (STAGE 5 & 6)")
print("="*60)
print(df_results.to_string(index=False, formatters={
    'Accuracy': '{:,.2%}'.format,
    'F1-Score': '{:,.2%}'.format
}))
print("="*60)

🏆           MODEL COMPARISON REPORT (STAGE 5 & 6)
     Vectorization               Model Accuracy F1-Score
Bag of Words (BoW)         Naive Bayes   82.88%   88.62%
Bag of Words (BoW) Logistic Regression   83.00%   89.04%
            TF-IDF         Naive Bayes   82.30%   89.36%
            TF-IDF Logistic Regression   84.42%   90.27%
